In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

In [2]:
#carichiamo il csv con i soggetti di Wikidata
df_wikidata = pd.read_csv("wd_dataset.csv", encoding="utf-8")

df_wikidata.head(3)


,entity,label,aliases,description,category
0,http://www.wikidata.org/entity/Q279782,Abarbarea,NaN,naiade della mitologia greca,greek_deity
1,http://www.wikidata.org/entity/Q2370052,Acanto,NaN,"personaggio della mitologia greca, probabilmen...",greek_deity
2,http://www.wikidata.org/entity/Q4668463,Acaste,NaN,Oceanina nella mitologia greca,greek_deity


In [3]:
#estraggo tutte le label di wikidata 
labels = (
    df_wikidata["label"]
    .dropna() #tolgo i valori nulli
    .str.strip() #tolgo gli spazi bianchi all'inizio e alla fine
    .tolist() #converto in lista

)

#estraggo gli aliases di wikidata separando quelli multipli
aliases = []
for alias_str in df_wikidata["aliases"].dropna(): #per ogni stringa di alias che non è nulla
    alias_list = [alias.strip() for alias in alias_str.split(",")] #separo gli alias con la virgola, tolgo gli spazi e metto tutto in minuscolo
    aliases.extend(alias_list) #aggiungo alla lista totale degli alias

#unisco tutto togliendo i duplicati
myth_terms = list(set(labels + aliases)) 
len(myth_terms)

2949

In [4]:
#ENDPOINT SPARLQ DI ARCO 
arco_endpoint = "https://dati.cultura.gov.it/sparql"

sparql_arco = SPARQLWrapper(arco_endpoint)
sparql_arco.setReturnFormat(JSON)

In [5]:
# NUMERO TOTALE DEI BENI STORICO ARTISTICI - senza duplicati 
query1 = """
PREFIX arco: <https://w3id.org/arco/ontology/arco/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 

SELECT (COUNT(DISTINCT ?item) AS ?totalItems)
WHERE {
  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .
}
"""


sparql_arco.setQuery(query1)
results = sparql_arco.query().convert()

for result in results["results"]["bindings"]:
    print(f"Total Historic or Artistic Properties: {result['totalItems']['value']}")

Total Historic or Artistic Properties: 2259266


In [6]:
#query pr estrarre subject e label dei beni storico artistici in batch (altrimenti esplode)
# Iterare su tutti i 2.258.640 beni con batch di 20000 (altrimenti esplode)

rows = []
batch_size = 20000
total_items = 2258640

for offset in range(0, total_items, batch_size):
    print(f"Processando batch: offset={offset}, items processati={len(rows)}")
    
    query2 = f"""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX a-cd: <https://w3id.org/arco/ontology/context-description/>

SELECT ?item ?subject ?label
WHERE {{

  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .
  
  OPTIONAL {{ ?item rdfs:label ?label . }}

  OPTIONAL {{ ?item a-cd:subject ?subject . }}

}}
LIMIT {batch_size}
OFFSET {offset}
"""
    
    sparql_arco.setQuery(query2)
    results = sparql_arco.query().convert()
    
    # estrai i dati da questo batch
    for r in results["results"]["bindings"]:
        rows.append({
            "item": r["item"]["value"] if "item" in r else None,
            "label": r["label"]["value"] if "label" in r else None,
            "subject": r["subject"]["value"] if "subject" in r else None
        })
    
    # se il batch è vuoto, ferma il loop
    if len(results["results"]["bindings"]) < batch_size:
        print(f"Batch incompleto ({len(results['results']['bindings'])} items), fine loop")
        break


Processando batch: offset=0, items processati=0
Processando batch: offset=20000, items processati=20000
Processando batch: offset=40000, items processati=40000
Processando batch: offset=60000, items processati=60000
Processando batch: offset=80000, items processati=80000
Processando batch: offset=100000, items processati=100000
Processando batch: offset=120000, items processati=120000
Processando batch: offset=140000, items processati=140000
Processando batch: offset=160000, items processati=160000
Processando batch: offset=180000, items processati=180000
Processando batch: offset=200000, items processati=200000
Processando batch: offset=220000, items processati=220000
Processando batch: offset=240000, items processati=240000
Processando batch: offset=260000, items processati=260000
Processando batch: offset=280000, items processati=280000
Processando batch: offset=300000, items processati=300000
Processando batch: offset=320000, items processati=320000
Processando batch: offset=340000

In [7]:

# creiamo il dataframe consolidato
df_arco = pd.DataFrame(rows)
df_arco = df_arco.drop_duplicates(subset="item")

print(f"Righe totali in df_arco : {len(df_arco)}")
df_arco[["item", "label", "subject"]].head(1)

Righe totali in df_arco : 1120196


,item,label,subject
0,https://w3id.org/arco/resource/HistoricOrArtis...,Arlecchino primordiale. Fine del cinquecento. ...,"figurino per costume di scena, maschile"


In [8]:
#ESTRAIAMO SOLO I SOGGETTI MITOLOGICI 

import re

# 1. filtro base --> togliamo i NaN
df_arco = df_arco[df_arco["subject"].notna()].copy()

#creiamo la nostra lista di termini myth_terms
terms = [
    re.escape(t)
    for t in myth_terms
    if isinstance(t, str)
    and t.strip() != ""
    and t[0].isupper()  
]

# regex unica case-sensitive e per evitare termini composti
pattern = r'(?<![A-Za-zÀ-ÿ])(' + '|'.join(terms) + r')(?![A-Za-zÀ-ÿ])'
regex = re.compile(pattern)


#TERMINI DA ACCETTARE SOLO SE COMPAIONO INSIEME A QUESTE PAROLE []
context_rules = {
    "Ascanio" : ["fuoco", "cervo"],"Callisto": ["Arcade", "Diana"],"Concordia": ["Allegoria", "Innocenza", "Giustizia"],
    "Cornacchia": ["metamorfosi"],"Egeo": ["Teseo"],"Elena": ["ratto"],"Ettore": ["morte", "Protesilao", "Achille", "Andromaca", "Paride"],
    "Europa": ["ratto", "Toro", "sibilla", "mito","allegoria","ancelle","leone"],
    "Grazie" : ["tre", "danzanti"],"Ida" : ["Linceo"],"Ippolito":["episodi", "caduta", "morte"],
    "Nilo":["personificazione","allegoria"],"Roma":["dea"],"Silvano":["Dio"],"Sonno":["allegoria"],
    "Vesta":["Giove"],"Vittoria":["allegoria"]
}


# TERMINI DA ESCLUDERE SE COMPAIONO CON DETERMINATE PAROLE
exclusion_rules = {
    "Abbondanza": ["Santa", "Sant","san","papa"],"Amore": ["Dio", "Madonna", "castello", "fontana","antica"],"Andromeda":["galassia"],"Apollo":["sala","tempio"],
    "Atlante":["Scienze","turisti"],"Carità": ["Santa", "Sant", "Maria"],
    "Carita": ["Santa", "Sant", "Maria"],"Diana" : ["Beata", "duca","scenografia","tempio","conte","san"],
    "Dioniso": ["stemma","stemmi", "san", "santi", "s."],"Dionisio":["Ritratto","san","barletta","beati","vescovo","Santi","stemma","papa","padre","beato"],
    "Elio" : ["ponte","ritratto","paesaggio", "veduta"],"Enea": ["Beato", "discepolo","stemma", "papa", "silvio"],
    "Ercole": ["Ricotti", "Dandini", "Comini", "Farnese","cardinale", "tempio", "stemma", "ritratto", "Roccotti", "colonne", "porto", "basilica", "santa"],
    "Ermes" : ["ritratto"],"Esculapio": ["tempio"],
    "Fede" : ["santi", "sant", "san", "dottrina", "biblici", "Francia", "cattolica", "madonna", "evangelisti", "religione", "dio", "angeli", "angioletti","santa", "papa", "santo", "cristiana", "chiesa", "miracolo", "misericordia","virtù", "ritratto", "stemma"],
    "Flora" :["ritratto", "sante", "circo", "ruota", "iside", "madonna", "tempio"],"Galatea":["ritratto"],
    "Giasone":["san","ritratto"],"Giove":["san","tempio"],
    "Giunone":["scenografia", "tempio"],"Giustizia" : ["porta", "palazzo", "misericordia", "san", "santa", "papa", "imperatore", "papale","Virtù", "apostoli","papa","Giglio"],
    "Gorgone" : ["santa"],"greco":["testo","teatro"],"Io": ["ritratti "],"Ippolito":["Nievo"],"Lavinia":["ritratto", "autoritratto"],
    "Leandro" : ["autoritratto","ritratto","stemma","San", "busto", "torre"],"Marte":["tempio","sala","ultore"],
    "Medea":["Colleoni"],"Mercurio":["tempio","san","assiso"],"Mirra":["San"],"Minerva":["autoritratto","chiesa","tempio"],
    "Narciso":["san"],"Nereo":["ritratto","san","santi","SS."],"Nettuno":["veduta"],"Oceano":["Atlantico"],
    "Oreste":["Ritratto","autoritratto"],"Pan":["zucchero","veduta"],"Patroclo":["san"],"Pirro":["Stipiciano"],"Pluto":["Topolino","Disney"],
    "Polidoro":["ritratto","Vicolo"],"Polissena":["Ritratto","santa"],"Priamo":["san","santo","Sulis","Fumagalli"],"Scilla":["topografica","veduta","paesaggio"],
    "Telemaco":["stemma","ritratto"],"Teseo":["Spirito"],"Tolomeo":["San","Santi","Ritratto","beato"],"Urano":["Monte"],
    "Venere":["Giorgio","Castello","modella","tempio"],"Verità":["stemma","Pierantonio","ritratto"],"Vittorie":["reggitarga"],
    "Speranza":["ritratto","santa","chiesa","papa","san","santo","madonna"],"Scilla":["ritratto"],"Serapide":["tempio"],"Achille":["stemma","ritratto","duomo","san"],
    "Adone":["sant"],"Cibele":["tempio"],"Cirene":["simone","palazzo"],"Teseo":["tempio","ritratto"],"Titano":["san"],"Ulisse":["ritratto","Dini","cardinalizia"],"Zeus":["tempio"]

}


def validate_matches(row):
    subject = row["subject"]
    valid_matches = []

    for match in row["matched_term"]:

        # PRIMO CHECK: controlla se il termine è nelle exclusion_rules
        if match in exclusion_rules:
            exclusion_keywords = exclusion_rules[match]
            # se almeno una parola di esclusione è presente, scarta il termine
            if any(re.search(r'\b' + re.escape(k) + r'\b', subject, re.IGNORECASE) for k in exclusion_keywords):
                continue  
        
        # SECONDO CHECK: controlla context_rules 
        # se il termine non ha regole → tienilo
        if match not in context_rules:
            valid_matches.append(match)
            continue

        # controlla parole contestuali se il match è ambiguo 
        keywords = context_rules[match]

        if any(re.search(r'\b' + re.escape(k) + r'\b', subject, re.IGNORECASE) for k in keywords):
            valid_matches.append(match)

    return valid_matches


df_arco["matched_term"] = df_arco["subject"].str.findall(regex) #trova tutti i termini e restituisce la lista
df_arco["matched_term"] = df_arco.apply(validate_matches, axis=1) #fa i controlli definiti sopra 

# df che risulta
df_matches = df_arco[df_arco["matched_term"].str.len() > 0]

In [9]:
# Verifica i risultati
print(f"Totale beni Arco: {len(df_arco)}")
print(f"Beni con match mitologico: {len(df_matches)}")

Totale beni Arco: 522846
Beni con match mitologico: 9519


In [107]:
#ulteriore iterazione NON case sensitive 
extra_check_terms = [
    "Aracne", "Argo", "Argonauti", "Arpia", "Arpie","Ascanio", "Callisto", "Centauro", "Cerva di Cerinea", "ciclope", "ciclopi", 
    "Chimera", "Cinghiale calidonio", "Cinghiale di Erimanto","Cupido", "Didone", "Elena", "Ippocampo",
    "Ippolito", "idra", "Minotauro", "musa", "muse", "putto", "putti", "Ratto di Europa","Sibilla samia", "Sibilla eritrea", "Sibilla tiburtina","Sirena", "Siringa", "Tritone", "satiro", "satiri"
]


extra_terms = [re.escape(t) for t in extra_check_terms]  #mette in sicurezza le parole per la regex 

extra_pattern = r'(?<![A-Za-zÀ-ÿ])(' + '|'.join(extra_terms) + r')(?![A-Za-zÀ-ÿ])'  
extra_regex = re.compile(extra_pattern, re.IGNORECASE)  #non case sensitive

#creo una copia del dataframe
df_matches = df_matches.copy()

# converte lista in set per confronto veloce
df_matches["matched_term_set"] = df_matches["matched_term"].apply(
    lambda x: set(x) if isinstance(x, list) else set()
)

#nuovo check
def find_new_matches(row):
    subject = row["subject"]
    already_found = row["matched_term_set"]

    all_hits = set(extra_regex.findall(subject))

    # rimuove quelli già trovati nel primo passaggio
    new_hits = all_hits - already_found

    return list(new_hits)


#applichiamo la funzione e creiamo una nuova colonna con gli extra_matched_terms
df_matches["extra_matched_terms"] = df_matches.apply(find_new_matches, axis=1)

#teniamo solo le righe utili
df_extra_hits = df_matches[df_matches["extra_matched_terms"].str.len() > 0].copy()


In [108]:
#normalizziamo i termini per poterli confrontare 
def normalize(s):
    return s.lower().strip() if isinstance(s, str) else ""

def merge_terms(row):
    base = row["matched_term"] if isinstance(row["matched_term"], list) else []
    extra = row["extra_matched_terms"] if isinstance(row["extra_matched_terms"], list) else []

    base_norm = {normalize(x): x for x in base}
    extra_norm = {normalize(x): x for x in extra}

    final = set()

    # 1. tengo gli extra che sono più specifici (es. europa --> ratto d'europa)
    for k, v in extra_norm.items():
        final.add(v)

    for b_key, b_val in base_norm.items():
        is_redundant = False

        for e_key in extra_norm:
            if b_key in e_key and b_key != e_key:
                is_redundant = True
                break

        if not is_redundant:
            final.add(b_val)  #tengo i termini base solo se non c'è una versione più specifica

    return list(final)

#applichiamo la funzione per avere i matched_term completi e più specifici
df_matches["final_terms"] = df_matches.apply(merge_terms, axis=1)

#df_matches[["subject", "final_terms"]].head()
df_checked = df_matches.sort_values(by="final_terms")[["item","label","subject", "final_terms"]]


In [109]:
print(len(df_checked))

9519


In [110]:
#esportiamo i risultati in un file csv

df_checked.to_csv("arco_myth_matches.csv", index=False, encoding="utf-8")


In [111]:
arco_endpoint = "https://dati.cultura.gov.it/sparql"
sparql_arco = SPARQLWrapper(arco_endpoint)
sparql_arco.setReturnFormat(JSON)

df_base = pd.read_csv("arco_myth_matches.csv", encoding="utf-8")
items = df_base["item"].dropna().unique().tolist() #estraggo la lista di item unici per poi fare ulteriori query su Wikidata

In [112]:
# funzione per eseguire query in batch su SPARQL

def run_batched_query(items_list, query_template, batch_size=100):

    all_dfs = []

    for i in range(0, len(items_list), batch_size):

        batch = items_list[i:i + batch_size]
        formatted_items = " ".join(f"<{uri}>" for uri in batch) 

        query = query_template.replace("{ITEMS}", formatted_items)

        sparql_arco.setQuery(query)

        try:
            results = sparql_arco.query().convert()

            rows = []
            for r in results["results"]["bindings"]:
                rows.append({k: v["value"] for k, v in r.items()})

            if rows:
                all_dfs.append(pd.DataFrame(rows))

        except Exception as e:
            print(f"Errore batch {i}: {e}")

    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)

    return pd.DataFrame()

In [113]:
#QUERY 1: identifier, creators, types

query_ict= """ 
        PREFIX arco: <https://w3id.org/arco/ontology/arco/>
        PREFIX a-dd: <https://w3id.org/arco/ontology/denotative-description/>
        PREFIX a-loc: <https://w3id.org/arco/ontology/location/>
        PREFIX dc: <http://purl.org/dc/elements/1.1/>
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        
        SELECT ?item ?identifier      
            (GROUP_CONCAT(DISTINCT ?creator; separator=", ") AS ?creators) 
            (SAMPLE (?creatorL) AS ?creatorLabel) 

            (GROUP_CONCAT(DISTINCT ?type; separator=", ") AS ?types) 
            (SAMPLE (?typeL) AS ?typeLabel) 

            (GROUP_CONCAT(DISTINCT ?instituteOrSite; separator=", ") AS ?institutesOrSites)
            (SAMPLE (?instituteOrSiteL) AS ?institutesOrSiteLabel) 
            
            (GROUP_CONCAT(DISTINCT ?place; separator=", ") AS ?coverage)
            (SAMPLE (?placeL) AS ?coverageLabel)
        
        WHERE {
            VALUES ?item {{ITEMS}} .
            ?item arco:uniqueIdentifier ?identifier .
 
            OPTIONAL {
                ?item dc:creator ?creator . 
                OPTIONAL {?creator rdfs:label ?creatorL .}
                }

            OPTIONAL {
                ?item a-dd:hasCulturalPropertyType ?type . 
                OPTIONAL {
                    ?type rdfs:label ?typeL .
                    FILTER(LANG(?typeL) = "it")
                    FILTER (!CONTAINS(STR(?typeL), "Tipo del bene:"))       
                }
            } 

            OPTIONAL {
                ?item a-loc:hasCulturalInstituteOrSite ?instituteOrSite . 
                OPTIONAL {?instituteOrSite rdfs:label ?instituteOrSiteL .}
                }

            OPTIONAL {
                ?item dc:coverage ?place .
                OPTIONAL {?place rdfs:label ?placeL .}
            } 
        }    
        
 
        GROUP BY ?item ?identifier
        """

In [114]:
#Q2 creation - date range 

query_date = """ 
        PREFIX arco: <https://w3id.org/arco/ontology/arco/>
        PREFIX a-cd: <https://w3id.org/arco/ontology/context-description/>

        SELECT ?item 
            (SAMPLE(STR(?event)) AS ?event)
            (SAMPLE(STR(?startDate)) AS ?start) 
            (SAMPLE(STR(?endDate)) AS ?end)
        WHERE {
            VALUES ?item 
                {{ITEMS}}
    
        OPTIONAL {
            ?item a-cd:hasDating ?dating .
            ?dating a-cd:hasDatingEvent ?event .
            ?event a-cd:specificTime ?time .
            ?time arco:startTime ?startDate ;
              arco:endTime ?endDate .
        
            # Filtro per assicurarci di prendere solo la creazione
            FILTER(REGEX(STR(?event), "creation", "i"))
        }        
    }
    GROUP BY ?item 
        """

In [115]:
#QUERY 3 - geometries and locations 
query_location = """ 
        PREFIX a-loc: <https://w3id.org/arco/ontology/location/>
        PREFIX clvapit: <https://w3id.org/italia/onto/CLV/>
        SELECT ?item
            (SAMPLE(STR(?geometry)) AS ?geometry) 
            (SAMPLE(STR(?lat)) AS ?lat)
            (SAMPLE(STR(?long)) AS ?long)
        WHERE {
            VALUES ?item {{ITEMS}} .
            # OPTIONAL per coordinate geografiche
            OPTIONAL {
                ?item clvapit:hasGeometry ?geometry .
                ?geometry a-loc:hasCoordinates ?coordinates .
                ?coordinates a-loc:lat ?lat ;
                            a-loc:long ?long .
            }        
        }
        GROUP BY ?item 
        """

In [116]:
#QUERY 4 - images 
query_img = """ 
        PREFIX foaf: <http://xmlns.com/foaf/0.1/>
 
        SELECT ?item (SAMPLE(?image) AS ?sampleImage) #  solo una tra quelle disponibili
        WHERE {
            VALUES ?item {{ITEMS}} .
            OPTIONAL {?item foaf:depiction ?image . }        
        }
	GROUP BY ?item
        """

In [117]:
#eseguo le query 
df_ict = run_batched_query(items, query_ict, batch_size=50)


In [118]:
df_date = run_batched_query(items, query_date,batch_size=50)

In [119]:
df_location = run_batched_query(items, query_location, batch_size=50)

In [120]:
df_img = run_batched_query(items, query_img, batch_size=50)

In [121]:
#creo un df unico 
df_enriched = df_base \
    .merge(df_ict, on="item", how="left") \
    .merge(df_date, on="item", how="left") \
    .merge(df_location, on="item", how="left") \
    .merge(df_img, on="item", how="left")


In [122]:
df_enriched.to_csv("arco_myth_enriched.csv", index=False)
print(len(df_enriched))

9519


In [4]:
#VISUALIZZIAMO IL DF ENRICHED 

df =pd.read_csv("arco_myth_enriched.csv")
df.head(1)

,item,label,subject,final_terms,identifier,creators,types,typeLabel,institutesOrSites,institutesOrSiteLabel,coverage,creatorLabel,event,start,end,geometry,lat,long,sampleImage
0,https://w3id.org/arco/resource/HistoricOrArtis...,Abbondanza (gemma) - produzione italiana (secc...,Abbondanza,['Abbondanza'],0800285568,NaN,https://w3id.org/arco/resource/CulturalPropert...,gemma,https://w3id.org/arco/resource/CulturalInstitu...,Galleria Estense,Modena (MO),NaN,https://w3id.org/arco/resource/Event/080028556...,(?) 1500,1699,https://w3id.org/arco/resource/Geometry/080028...,44.646037,10.934481,http://www.sigecweb.beniculturali.it/images/fu...


In [2]:
#VISUALIZZIAMO SOLO LE LABEL DEL DF 

df = pd.read_csv("arco_myth_enriched.csv")

colonne_selezionate = [
    "item",
    "label",
    "subject",
    "final_terms",
    "typeLabel",
    "creatorLabel",
    "institutesOrSiteLabel",
    "coverage",
    "start",
    "end"
]

df_label = df[colonne_selezionate].copy()
df_label.head(50)

,item,label,subject,final_terms,typeLabel,creatorLabel,institutesOrSiteLabel,coverage,start,end
0,https://w3id.org/arco/resource/HistoricOrArtis...,Abbondanza (dipinto) - ambito napoletano (sec....,Abbondanza,['Abbondanza'],dipinto,NaN,NaN,Faicchio (BN),1700,1799
1,https://w3id.org/arco/resource/HistoricOrArtis...,"Pace, Abbondanza e Concordia (disegno) by Nani...","Pace, Abbondanza e Concordia",['Abbondanza'],disegno,Nanin Pietro - 1808/ 1889,Museo Civico di Castelvecchio,Verona (VR),1840,1860
2,https://w3id.org/arco/resource/HistoricOrArtis...,Abbondanza e Misericordia (dipinto) by Lama Gi...,Abbondanza e Misericordia,['Abbondanza'],dipinto,Lama Giovan Battista - 1673 ca./ 1748,NaN,Napoli (NA),ca 1730,ca 1732
3,https://w3id.org/arco/resource/HistoricOrArtis...,"Abbondanza (dipinto, opera isolata) - ambito I...",Abbondanza,['Abbondanza'],dipinto,NaN,Galleria nazionale dell'Umbria,Perugia (PG),ca 1700,ca 1749
4,https://w3id.org/arco/resource/HistoricOrArtis...,Abbondanza (scultura) di Della Valle Filippo (...,Abbondanza,['Abbondanza'],scultura,Della Valle Filippo (attribuito),Museo Nazionale del Palazzo di Venezia,Roma (RM),1759,1760
5,https://w3id.org/arco/resource/HistoricOrArtis...,"Abbondanza (statuetta, opera isolata) - ambito...",Abbondanza,['Abbondanza'],statuetta,NaN,Galleria nazionale dell'Umbria,Perugia (PG),CA 1590,CA 1610
6,https://w3id.org/arco/resource/HistoricOrArtis...,"Abbondanza (dipinto, elemento d'insieme) di Na...",Abbondanza,['Abbondanza'],dipinto,Natali Giovanni Battista - 1630 ca./ 1696,NaN,Cella Dati (CR),post 1650,ante 1699
7,https://w3id.org/arco/resource/HistoricOrArtis...,"L'Abbondanza (rilievo, opera isolata) di Berna...",L'Abbondanza,['Abbondanza'],rilievo,Bernasconi Tommaso - notizie 1907,NaN,San Pellegrino Terme (BG),1907,1907
8,https://w3id.org/arco/resource/HistoricOrArtis...,"Abbondanza (dipinto, elemento d'insieme) di Ga...",Abbondanza,['Abbondanza'],dipinto,Gandolfi Tommaso - notizie 1657-1710,NaN,Ferrara (FE),1690,1710
9,https://w3id.org/arco/resource/HistoricOrArtis...,Abbondanza (statuetta) - ambito di Augusta (me...,Abbondanza,['Abbondanza'],statuetta,NaN,Museo Nazionale del Palazzo di Venezia,Roma (RM),1540,1560


In [3]:
# --------------- codice per estrazione properties wikidata -------------- #

# pulizia matched terms
def clean_terms(term):
    cleaned_term = term.strip("[").strip("]")
    return cleaned_term

# sto prendendo dal df completo di arco_myth_enriched
terms_to_match = df["final_terms"].apply(clean_terms)
print(terms_to_match.head)
# set dei termini matchati in arco
set_of_matched_terms = set()

for idx, term_list in terms_to_match.items():
    print("Processing item", idx)
    if isinstance(term_list, str):
        for term in term_list.split(","):
            cleaned_element = term.strip().strip("'").strip()
            # per evitare che stesso nome con maiuscola/minuscola sporchi il set
            cleaned_cap = cleaned_element.capitalize() if cleaned_element.islower() else cleaned_element
            set_of_matched_terms.add(cleaned_cap)

print("the set of terms:\n", set_of_matched_terms)
print("The set counts", len(set_of_matched_terms), "terms")

<bound method NDFrame.head of 0                             'Abbondanza'
1                             'Abbondanza'
2                             'Abbondanza'
3                             'Abbondanza'
4                             'Abbondanza'
                       ...                
9511    'sibilla Samia', 'sibilla eritrea'
9512                 'sirena', 'Partenope'
9513                      'siringa', 'Pan'
9514                      'siringa', 'Pan'
9515                      'siringa', 'Pan'
Name: final_terms, Length: 9516, dtype: object>
Processing item 0
Processing item 1
Processing item 2
Processing item 3
Processing item 4
Processing item 5
Processing item 6
Processing item 7
Processing item 8
Processing item 9
Processing item 10
Processing item 11
Processing item 12
Processing item 13
Processing item 14
Processing item 15
Processing item 16
Processing item 17
Processing item 18
Processing item 19
Processing item 20
Processing item 21
Processing item 22
Processing item 23
Pro

In [4]:
# estrazione entità wikidata (filtro su label e alias)
wd_dataset = pd.read_csv("wd_dataset.csv", encoding="utf-8")
#print(wd_dataset.head)

wd_matched_entities = set()

for term in set_of_matched_terms:
    #print("I am looking for", term)
    check = (wd_dataset['label'] == term) | (wd_dataset['aliases'] == term)
    
    entities_to_extract = wd_dataset.loc[check, 'entity'].tolist()
    print("The match for", term, "is", entities_to_extract)
    
    # aggiungi entity al set
    for entity in entities_to_extract:
        if entity:
            wd_matched_entities.add(entity)

print("The set of extracted entities:\n", wd_matched_entities)
print("The set of extracted entities counts:", len(wd_matched_entities), "uris")

wd_matched_entities_list = list(wd_matched_entities)

# ne abbiamo di più perché guarda anche aliases (?)


The match for Licurgo is ['http://www.wikidata.org/entity/Q368157', 'http://www.wikidata.org/entity/Q270117', 'http://www.wikidata.org/entity/Q211326', 'http://www.wikidata.org/entity/Q609650', 'http://www.wikidata.org/entity/Q10317982']
The match for Archemoro is ['http://www.wikidata.org/entity/Q1419140']
The match for Eroti is ['http://www.wikidata.org/entity/Q1361380']
The match for Andromaca is ['http://www.wikidata.org/entity/Q19376']
The match for Cibele is ['http://www.wikidata.org/entity/Q188236']
The match for Pallante is ['http://www.wikidata.org/entity/Q457294', 'http://www.wikidata.org/entity/Q2463695', 'http://www.wikidata.org/entity/Q429306', 'http://www.wikidata.org/entity/Q3892261', 'http://www.wikidata.org/entity/Q2048018']
The match for Odisseo is []
The match for Arione is ['http://www.wikidata.org/entity/Q842435']
The match for Selene is ['http://www.wikidata.org/entity/Q131585']
The match for Eurialo is ['http://www.wikidata.org/entity/Q16647668', 'http://www.wiki

In [5]:
# set wikidata endpoint
wd_endpoint = "https://query.wikidata.org/bigdata/namespace/wdq/sparql" # sui tutorial c'è questo
# set endpoint
sparql_wd = SPARQLWrapper(wd_endpoint)

In [6]:
# function for wikidata queries
""" def extract_batch_properties(entities_list, query_prop, batch_size=3):
    all_properties = []

    for i in range(0, len(entities_list), batch_size):
        batch = entities_list[i:i + batch_size]
        formatted_items = " ".join(f"<{uri}>" for uri in batch) 

        query = query_prop.replace("{ENTITIES}", formatted_items)

        sparql_wd.setQuery(query_prop)
        sparql_wd.setReturnFormat(JSON)

        try:
            results = sparql_wd.query().convert()

            for r in results["results"]["bindings"]:
                # Controlliamo i dati reali restituiti da Wikidata
                property_uri = r["property"]["value"] if "property" in r else "N/A"

                # Il servizio aggiunge SEMPRE "Label" al nome della variabile principale
                if (
                    "propertyLabel" in r
                    and r["propertyLabel"]["value"] != property_uri
                ):
                    label = r["propertyLabel"]["value"]
                else:
                    # Se il label service fallisce, estraiamo l'ID finale (es. P31) come fallback
                    label = property_uri.split("/")[-1]

                all_properties.append({"property": property_uri, "label": label})

        except Exception as e:
            print(f"Errore: {e}")
            return []

        return all_properties """

' def extract_batch_properties(entities_list, query_prop, batch_size=3):\n    all_properties = []\n\n    for i in range(0, len(entities_list), batch_size):\n        batch = entities_list[i:i + batch_size]\n        formatted_items = " ".join(f"<{uri}>" for uri in batch) \n\n        query = query_prop.replace("{ENTITIES}", formatted_items)\n\n        sparql_wd.setQuery(query_prop)\n        sparql_wd.setReturnFormat(JSON)\n\n        try:\n            results = sparql_wd.query().convert()\n\n            for r in results["results"]["bindings"]:\n                # Controlliamo i dati reali restituiti da Wikidata\n                property_uri = r["property"]["value"] if "property" in r else "N/A"\n\n                # Il servizio aggiunge SEMPRE "Label" al nome della variabile principale\n                if (\n                    "propertyLabel" in r\n                    and r["propertyLabel"]["value"] != property_uri\n                ):\n                    label = r["propertyLabel"]["value"]

In [7]:
# query estrazione proprietà wd a partire dal set di entities estratte
query_wd_properties = """ 
        SELECT DISTINCT ?property ?propertyLabel
        WHERE {
            # Prendiamo la relazione (wdt:...) e l'oggetto
            wd:Q11688446 ?propertyUri ?object .
            
            # Isoliamo solo le proprietà dirette
            FILTER(STRSTARTS(STR(?propertyUri), "http://www.wikidata.org/prop/direct/"))
            
            # Trasformiamo l'URI della relazione nell'URI dell'entità Proprietà (wd:P...)
            # Usiamo ?property come nome standard, così il SERVICE sa esattamente cosa cercare
            BIND(URI(REPLACE(STR(?propertyUri), "http://www.wikidata.org/prop/direct/", "http://www.wikidata.org/entity/")) AS ?property)
            
            # Il Label Service intercetta ?property e genera automaticamente ?propertyLabel
            SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
        }
        ORDER BY ?propertyLabel
        """

In [ ]:
# run query
#wd_properties = extract_batch_properties(wd_matched_entities_list, query_wd_properties, batch_size=3)

In [8]:
def extract_properties(query_prop):
    all_properties = []

    sparql_wd.setQuery(query_prop)
    sparql_wd.setReturnFormat(JSON)

    try:
        results = sparql_wd.query().convert()

        for r in results["results"]["bindings"]:
            # Controlliamo i dati reali restituiti da Wikidata
            property_uri = r["property"]["value"] if "property" in r else "N/A"

            # Il servizio aggiunge SEMPRE "Label" al nome della variabile principale
            if (
                "propertyLabel" in r
                and r["propertyLabel"]["value"] != property_uri
            ):
                label = r["propertyLabel"]["value"]
            else:
                # Se il label service fallisce, estraiamo l'ID finale (es. P31) come fallback
                label = property_uri.split("/")[-1]

            if "ID" not in label:
                all_properties.append({"property": property_uri, "label": label}) # se label contiene "ID" non inserire nè property nè label

    except Exception as e:
        print(f"Errore: {e}")
        return []

    return all_properties

wd_properties = extract_properties(query_wd_properties)
print("Wikidata properties for one entity:", wd_properties)

Errore: <urlopen error [Errno 11001] getaddrinfo failed>
Wikidata properties for one entity: []
